# Uber Data Pipeline (Fixed Version) — Weather Enhanced

This notebook improves the original taxi ETL flow and adds historical weather integration from the **Open-Meteo Archive API**.

## What is improved
- Cleaner, reusable functions for each stage.
- Explicit schema checks and basic data quality filters.
- Timezone-safe timestamp handling (`America/New_York`).
- Hourly weather ingestion by borough anchor coordinates.
- Taxi + weather join at `pickup_hour x borough` grain.
- Final aggregated mart for analysis/dashboarding.


## 1) Imports and Settings

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

import pandas as pd
import requests

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

NY_TZ = "America/New_York"
OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

# Example input files (adjust as needed)
TAXI_PARQUET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
TAXI_ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"


## 2) Reusable Helpers

In [ ]:
@dataclass(frozen=True)
class BoroughPoint:
    borough: str
    latitude: float
    longitude: float


BOROUGH_POINTS: tuple[BoroughPoint, ...] = (
    BoroughPoint("Manhattan", 40.7831, -73.9712),
    BoroughPoint("Brooklyn", 40.6782, -73.9442),
    BoroughPoint("Queens", 40.7282, -73.7949),
    BoroughPoint("Bronx", 40.8448, -73.8648),
    BoroughPoint("Staten Island", 40.5795, -74.1502),
)


def _ensure_datetime(series: pd.Series, tz: str = NY_TZ) -> pd.Series:
    """Parse datetimes and localize/convert to a target timezone."""
    dt = pd.to_datetime(series, errors="coerce")
    if dt.dt.tz is None:
        return dt.dt.tz_localize(tz, nonexistent="shift_forward", ambiguous="NaT")
    return dt.dt.tz_convert(tz)


def _validate_required_columns(df: pd.DataFrame, required: Iterable[str], frame_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} is missing required columns: {missing}")


## 3) Taxi Ingestion + Cleaning

In [ ]:
def load_taxi_data(parquet_url: str) -> pd.DataFrame:
    df = pd.read_parquet(parquet_url)

    required = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "total_amount",
    ]
    _validate_required_columns(df, required, "taxi_df")
    return df


def clean_taxi_data(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["pickup_ts"] = _ensure_datetime(out["tpep_pickup_datetime"])
    out["dropoff_ts"] = _ensure_datetime(out["tpep_dropoff_datetime"])

    out["trip_duration_min"] = (out["dropoff_ts"] - out["pickup_ts"]).dt.total_seconds() / 60.0

    # Core sanity filters
    out = out.loc[
        out["pickup_ts"].notna()
        & out["dropoff_ts"].notna()
        & (out["trip_duration_min"] > 0)
        & (out["fare_amount"] >= 0)
        & (out["trip_distance"] > 0)
    ].copy()

    # Winsorize extreme outliers at 99.5 percentile for stability
    for col in ["trip_duration_min", "fare_amount", "trip_distance", "total_amount"]:
        upper = out[col].quantile(0.995)
        out[col] = out[col].clip(upper=upper)

    out["pickup_hour"] = out["pickup_ts"].dt.floor("H")
    out["pickup_date"] = out["pickup_ts"].dt.date
    out["pickup_dow"] = out["pickup_ts"].dt.day_name()
    out["is_weekend"] = out["pickup_ts"].dt.dayofweek >= 5

    return out


## 4) Taxi Zone Lookup (for borough mapping)

In [ ]:
def load_taxi_zone_lookup(csv_url: str) -> pd.DataFrame:
    zones = pd.read_csv(csv_url)
    _validate_required_columns(zones, ["LocationID", "Borough"], "zones_df")
    zones = zones[["LocationID", "Borough"]].drop_duplicates()
    zones = zones.rename(columns={"LocationID": "PULocationID", "Borough": "pickup_borough"})
    return zones


def attach_pickup_borough(taxi_df: pd.DataFrame, zones_df: pd.DataFrame) -> pd.DataFrame:
    merged = taxi_df.merge(zones_df, on="PULocationID", how="left")
    merged["pickup_borough"] = merged["pickup_borough"].fillna("Unknown")
    return merged


## 5) Open-Meteo Weather Ingestion + Processing

In [ ]:
def fetch_open_meteo_hourly(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    timezone: str = NY_TZ,
) -> pd.DataFrame:
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,snowfall,windspeed_10m,weathercode",
        "timezone": timezone,
    }
    response = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=60)
    response.raise_for_status()

    payload = response.json()
    hourly = payload.get("hourly", {})
    if not hourly or "time" not in hourly:
        return pd.DataFrame()

    weather_df = pd.DataFrame(hourly)
    weather_df["weather_ts_hour"] = pd.to_datetime(weather_df["time"], errors="coerce")

    rename_map = {
        "precipitation": "precipitation_mm",
        "rain": "rain_mm",
        "snowfall": "snowfall_cm",
        "windspeed_10m": "wind_speed",
    }
    weather_df = weather_df.rename(columns=rename_map)

    keep = [
        "weather_ts_hour",
        "temperature_2m",
        "precipitation_mm",
        "rain_mm",
        "snowfall_cm",
        "wind_speed",
        "weathercode",
    ]
    existing = [c for c in keep if c in weather_df.columns]
    weather_df = weather_df[existing].copy()

    for c in ["precipitation_mm", "rain_mm", "snowfall_cm"]:
        if c in weather_df.columns:
            weather_df[c] = weather_df[c].fillna(0)

    weather_df["is_rain"] = weather_df.get("rain_mm", 0) > 0
    weather_df["is_snow"] = weather_df.get("snowfall_cm", 0) > 0

    def to_severity(row: pd.Series) -> str:
        p = float(row.get("precipitation_mm", 0) or 0)
        s = float(row.get("snowfall_cm", 0) or 0)
        if p == 0 and s == 0:
            return "clear"
        if p < 2 and s < 1:
            return "light"
        if p < 8 and s < 3:
            return "moderate"
        return "severe"

    weather_df["weather_severity"] = weather_df.apply(to_severity, axis=1)
    return weather_df


def build_weather_for_boroughs(
    borough_points: Iterable[BoroughPoint],
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for point in borough_points:
        borough_weather = fetch_open_meteo_hourly(
            latitude=point.latitude,
            longitude=point.longitude,
            start_date=start_date,
            end_date=end_date,
        )
        if borough_weather.empty:
            continue
        borough_weather["pickup_borough"] = point.borough
        frames.append(borough_weather)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


## 6) Join Taxi and Weather

In [ ]:
def join_taxi_weather(taxi_df: pd.DataFrame, weather_df: pd.DataFrame) -> pd.DataFrame:
    required_taxi = ["pickup_hour", "pickup_borough"]
    required_weather = ["weather_ts_hour", "pickup_borough"]
    _validate_required_columns(taxi_df, required_taxi, "taxi_df")
    _validate_required_columns(weather_df, required_weather, "weather_df")

    merged = taxi_df.merge(
        weather_df,
        left_on=["pickup_hour", "pickup_borough"],
        right_on=["weather_ts_hour", "pickup_borough"],
        how="left",
    )
    return merged


def build_hourly_borough_mart(joined_df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        joined_df.groupby(["pickup_hour", "pickup_borough", "weather_severity"], dropna=False)
        .agg(
            trip_count=("pickup_ts", "size"),
            avg_fare=("fare_amount", "mean"),
            avg_trip_duration_min=("trip_duration_min", "mean"),
            avg_trip_distance=("trip_distance", "mean"),
            avg_precipitation_mm=("precipitation_mm", "mean"),
            avg_temperature_2m=("temperature_2m", "mean"),
        )
        .reset_index()
        .sort_values(["pickup_hour", "pickup_borough"])
    )
    return grouped


## 7) Run Pipeline (Example: 2023-01)

> This cell runs the full flow end-to-end. If network access is limited, execute each step separately and cache files locally.

In [ ]:
START_DATE = "2023-01-01"
END_DATE = "2023-01-31"

raw_taxi_df = load_taxi_data(TAXI_PARQUET_URL)
cleaned_taxi_df = clean_taxi_data(raw_taxi_df)

zones_df = load_taxi_zone_lookup(TAXI_ZONE_LOOKUP_URL)
taxi_with_borough_df = attach_pickup_borough(cleaned_taxi_df, zones_df)

weather_df = build_weather_for_boroughs(BOROUGH_POINTS, start_date=START_DATE, end_date=END_DATE)

trip_weather_df = join_taxi_weather(taxi_with_borough_df, weather_df)
hourly_mart_df = build_hourly_borough_mart(trip_weather_df)

print("Raw taxi rows:", len(raw_taxi_df))
print("Cleaned taxi rows:", len(cleaned_taxi_df))
print("Weather rows:", len(weather_df))
print("Joined rows:", len(trip_weather_df))
print("Hourly mart rows:", len(hourly_mart_df))

hourly_mart_df.head(10)


## 8) Optional Export

Uncomment and run if you want local outputs.

In [ ]:
# trip_weather_df.to_parquet('trip_weather_2023_01.parquet', index=False)
# hourly_mart_df.to_parquet('trip_weather_hourly_mart_2023_01.parquet', index=False)
